In [1]:
import pickle
import h5py
import os
import numpy as np
import json

In [2]:
path = "../FMM/test_/Dict/test_None.pkl"

# Pickle

In [3]:
size_bytes_pickle = os.path.getsize(path)
print("Pickle file size :", size_bytes_pickle)

Pickle file size : 587042603


# H5PY

In [4]:
data = pickle.load(open(path, "rb"))

In [5]:
parameters = data["parameters"]
# data["fsky"] = np.array(data["fsky"])
duration = data["duration"]
qubic_dict = data["qubic_dict"]

In [6]:
with h5py.File("data.h5", "w") as h5f:
    grp_params = h5f.create_group("parameters")
    grp_qubic_dict = h5f.create_group("qubic_dict")
    for key, value in data.items():
        print(key)
        print(type(value))
        if key == "parameters":
            grp_params.attrs["parameters"] = json.dumps(value)
        elif key == "qubic_dict":
            grp_qubic_dict.attrs["qubic_dict"] = json.dumps(value)
        elif key in ["fsky", "duration"]:
            h5f.attrs[key] = value
        else:
            h5f.create_dataset(key, data=value, compression="gzip")

maps_in
<class 'numpy.ndarray'>
maps_in_convolved
<class 'numpy.ndarray'>
maps
<class 'numpy.ndarray'>
maps_noise
<class 'numpy.ndarray'>
tod
<class 'numpy.ndarray'>
nus
<class 'numpy.ndarray'>
coverage
<class 'numpy.ndarray'>
convergence
<class 'numpy.ndarray'>
center
<class 'tuple'>
parameters
<class 'dict'>
fwhm_in
<class 'numpy.ndarray'>
fwhm_out
<class 'numpy.ndarray'>
fwhm_rec
<class 'numpy.ndarray'>
seenpix
<class 'numpy.ndarray'>
fsky
<class 'numpy.float64'>
duration
<class 'float'>
qubic_dict
<class 'dict'>


In [7]:
size_bytes_h5 = os.path.getsize("data.h5")
print("h5py file size :", size_bytes_h5)

h5py file size : 243408905


# Comparison

In [8]:
print("Pickle file size :", size_bytes_pickle, " bytes.")
print("h5py file size :", size_bytes_h5, " bytes.")
print("Difference : ", size_bytes_pickle - size_bytes_h5, " bytes.")
print("h5 saves ", np.round((1 - size_bytes_h5 / size_bytes_pickle) * 100, 2), "% of memory")

Pickle file size : 587042603  bytes.
h5py file size : 243408905  bytes.
Difference :  343633698  bytes.
h5 saves  58.54 % of memory


# Test Qhdf5

In [9]:
from qubic.lib.Qhdf5 import HDF5Dict

hdf5 = HDF5Dict()

hwloc/linux: Ignoring PCI device with non-16bit domain.
Pass --enable-32bits-pci-domain to configure to support such devices
(warning: it would break the library ABI, don't enable unless really needed).


In [10]:
hdf5.save_dict("test.h5", data)

In [11]:
dict_test = hdf5.load_dict("test.h5")

In [12]:
print(data.keys())
print(dict_test["fsky"])
print(dict_test["nus"])
print(dict_test["qubic_dict"])
print(dict_test["qubic_dict"].keys())
print(dict_test["parameters"])

dict_keys(['maps_in', 'maps_in_convolved', 'maps', 'maps_noise', 'tod', 'nus', 'coverage', 'convergence', 'center', 'parameters', 'fwhm_in', 'fwhm_out', 'fwhm_rec', 'seenpix', 'fsky', 'duration', 'qubic_dict'])
0.033665974934895836
[150.4153874  220.02080088  30.          44.          70.
 100.         143.         217.         353.        ]
{'debug': False, 'config': 'FI', 'filter_nu': 220000000000.0, 'filter_relative_bandwidth': 0.25, 'beam_shape': 'gaussian', 'MultiBand': True, 'nf_sub': 8, 'center_detector': False, 'psd': None, 'bandwidth': None, 'twosided': None, 'sigma': None, 'detector_nep': 4.7e-17, 'detector_fknee': 0, 'detector_fslope': 1, 'detector_ncorr': 10, 'detector_ngrids': 1, 'detector_tau': 0.01, 'polarizer': True, 'synthbeam_fraction': 1, 'synthbeam_kmax': 1, 'synthbeam_peak150_fwhm': 0.39268176, 'ripples': False, 'nripples': 0, 'focal_length': 0.3, 'optics': 'CalQubic_Optics_v3_CC_FFF.txt', 'primbeam': 'CalQubic_PrimBeam_v2.fits', 'detarray': 'CalQubic_DetArray_v4_C

# Test reliability between HDF5 and Pickle

In [13]:
planck143_pkl = pickle.load(open("../../../../../qubic/data/Planck143GHz.pkl", "rb"))["noise143"]

hdf5.save_array("planck_test.h5", planck143_pkl)
planck143_hdf5 = hdf5.load_array("planck_test.h5")

In [14]:
print(np.array_equal(planck143_pkl, planck143_hdf5))
print(np.mean(planck143_pkl - planck143_hdf5))

True
0.0
